# Weather Trend Forecasting — Basic Assessment (Ottawa)



## Scope
- **City:** Ottawa, Canada
- **Time feature:** `last_updated` (used to build a daily time series)
- **Targets (visualized & forecasted):**
  - Temperature (daily **average**)
  - Precipitation (daily **total**)

## Requirements Covered
1. **Data Cleaning & Preprocessing**: missing values, outliers, normalization  
2. **EDA**: trends, correlations, patterns  
3. **Visualizations**: temperature & precipitation  
4. **Model Building**: a basic forecasting model + evaluation metrics (time-based split)

**PM Accelerator Mission:**

> PM Accelerator’s mission is to break down financial barriers and advance educational fairness by making top‑tier education, resources, and tools accessible to individuals from any background.

Source: https://www.pmaccelerator.io/about-us


In [ ]:
# =========================
# 1) Imports & settings
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (14, 5)
pd.set_option("display.max_columns", 200)

from IPython.display import display


## 2) Load data

**Option A (recommended):** place `GlobalWeatherRepository.csv` in the project folder.  
**Option B:** download from Kaggle (requires Kaggle API credentials).

In [ ]:
# --- Option A: local file ---
DATA_PATH = "GlobalWeatherRepository.csv"
df = pd.read_csv(DATA_PATH)

# Quick look
print("Shape:", df.shape)
df.head()


## 3) Filter to Ottawa and build a daily time series

The raw dataset can contain **multiple observations per day** (different hours).  
To analyze daily trends, we aggregate to daily level:

- Temperature → **mean** (daily average)
- Precipitation → **sum** (daily total)

In [ ]:
# Ensure datetime
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

# Focus on Ottawa (case-insensitive)
ottawa_df = df[df["location_name"].str.lower().eq("ottawa")].copy()

print("Ottawa rows:", len(ottawa_df))
ottawa_df[["country", "location_name"]].drop_duplicates().head()


In [ ]:
# Create a pure date column and aggregate to daily granularity
ottawa_df["date"] = ottawa_df["last_updated"].dt.floor("D")

# Select columns we need (temperature + precipitation, plus a few context vars for EDA correlations)
cols_needed = [
    "date",
    "temperature_celsius",
    "precip_mm",
    "humidity",
    "pressure_mb",
    "wind_kph",
]
available = [c for c in cols_needed if c in ottawa_df.columns]
missing = sorted(set(cols_needed) - set(available))
print("Available columns:", available)
if missing:
    print("Missing columns (not found in dataset):", missing)

ottawa_daily = (
    ottawa_df[available]
    .groupby("date", as_index=False)
    .agg({
        "temperature_celsius": "mean",
        "precip_mm": "sum",
        "humidity": "mean" if "humidity" in available else "first",
        "pressure_mb": "mean" if "pressure_mb" in available else "first",
        "wind_kph": "mean" if "wind_kph" in available else "first",
    })
    .sort_values("date")
)

ottawa_daily.head()


## 4) Data Cleaning & Preprocessing

### 4.1 Handle missing dates / missing values
Time series models expect a consistent date index.  
We reindex to a complete daily range and fill gaps.

### 4.2 Handle outliers
For a basic assessment, we use an **IQR-based clipping** approach (winsorization).

### 4.3 Normalize
We keep a normalized copy of the target for modeling (optional but demonstrates the step).

In [ ]:
# 4.1 Reindex to a complete daily date range
ottawa_daily = ottawa_daily.set_index("date").asfreq("D")

# Check missingness
missing_counts = ottawa_daily.isna().sum()
print("Missing values per column:
", missing_counts)

# Fill missing values:
# - temperature: time interpolation (then forward/backward fill as a fallback)
# - precip: fill missing with 0 (no observation → assume 0 for daily total, safer for precip)
if "temperature_celsius" in ottawa_daily.columns:
    ottawa_daily["temperature_celsius"] = (
        ottawa_daily["temperature_celsius"]
        .interpolate(method="time")
        .ffill()
        .bfill()
    )
if "precip_mm" in ottawa_daily.columns:
    ottawa_daily["precip_mm"] = ottawa_daily["precip_mm"].fillna(0)

# For other numeric columns, forward fill then backfill
for c in ottawa_daily.columns:
    if c not in ["temperature_celsius", "precip_mm"]:
        ottawa_daily[c] = ottawa_daily[c].ffill().bfill()

# 4.2 Outlier handling via IQR clipping (winsorization)
def iqr_clip(s: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return s.clip(lower=lo, upper=hi)

for target in ["temperature_celsius", "precip_mm"]:
    if target in ottawa_daily.columns:
        ottawa_daily[target] = iqr_clip(ottawa_daily[target])

# 4.3 Normalization (Min-Max) for temperature target
scaler = MinMaxScaler()
ottawa_daily["temp_norm"] = scaler.fit_transform(ottawa_daily[["temperature_celsius"]])

ottawa_daily.head()


## 5) Exploratory Data Analysis (EDA)

We focus on:
- **Trends** over time (temperature & precipitation)
- **Patterns** (e.g., seasonality visible in the plot)
- **Correlations** between key numeric variables

In [ ]:
# 5.1 Temperature trend (daily average)
ottawa_daily["temperature_celsius"].plot()
plt.title("Ottawa — Daily Average Temperature (°C)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.show()

# 5.2 Precipitation trend (daily total)
ottawa_daily["precip_mm"].plot()
plt.title("Ottawa — Daily Total Precipitation (mm)")
plt.xlabel("Date")
plt.ylabel("Precipitation (mm)")
plt.show()


In [ ]:
# 5.3 Correlations (basic)
corr_cols = [c for c in ["temperature_celsius", "precip_mm", "humidity", "pressure_mb", "wind_kph"] if c in ottawa_daily.columns]
corr = ottawa_daily[corr_cols].corr()

print("Correlation matrix:")
display(corr)

# Simple heatmap without seaborn
plt.figure(figsize=(6, 5))
plt.imshow(corr.values, aspect="auto")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar()
plt.title("Correlation Heatmap (Ottawa, daily)")
plt.tight_layout()
plt.show()


## 6) Model Building — Basic Forecasting

We build a simple forecasting baseline using **lag-1** (yesterday's value).  
This is a standard baseline for time series and is suitable for the basic assessment.

### Train/Test split
We use an **80/20 time-based split** (no shuffling).

### Metrics
- MAE
- RMSE
- MAPE (optional; reported carefully)

In [ ]:
# Prepare target series
y = ottawa_daily["temperature_celsius"].copy()

# Time-based split (80/20)
split_idx = int(len(y) * 0.8)
train_y = y.iloc[:split_idx]
test_y  = y.iloc[split_idx:]

# Naive baseline: predict today's temperature as yesterday's temperature
pred = test_y.shift(1)

# Align to remove the first NaN prediction
y_true = test_y.iloc[1:]
y_pred = pred.iloc[1:]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# MAPE can be unstable if values are close to 0; temperature usually OK but still handle safely
eps = 1e-6
mape = np.mean(np.abs((y_true - y_pred) / (np.maximum(np.abs(y_true), eps)))) * 100

print(f"Naive baseline (lag-1) — Temperature")
print(f"MAE : {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAPE: {mape:.2f}%")


In [ ]:
# Plot actual vs predicted (test period)
plt.figure(figsize=(14, 5))
plt.plot(test_y.index, test_y.values, label="Actual")
plt.plot(y_pred.index, y_pred.values, label="Predicted (lag-1)")
plt.title("Ottawa — Forecast (Naive Baseline) on Test Period")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.show()


## 7) (Optional) Baseline forecast for precipitation

Same approach: daily precipitation forecasted using lag-1.  
Because precipitation can be sparse (many zeros), interpret MAPE cautiously.

In [ ]:
y_p = ottawa_daily["precip_mm"].copy()

split_idx = int(len(y_p) * 0.8)
test_p = y_p.iloc[split_idx:]

pred_p = test_p.shift(1)
p_true = test_p.iloc[1:]
p_pred = pred_p.iloc[1:]

mae_p = mean_absolute_error(p_true, p_pred)
rmse_p = np.sqrt(mean_squared_error(p_true, p_pred))

# MAPE for precip can explode when true values are 0; we report MAE/RMSE as primary
print(f"Naive baseline (lag-1) — Precipitation")
print(f"MAE : {mae_p:.3f}")
print(f"RMSE: {rmse_p:.3f}")

plt.figure(figsize=(14, 5))
plt.plot(test_p.index, test_p.values, label="Actual")
plt.plot(p_pred.index, p_pred.values, label="Predicted (lag-1)")
plt.title("Ottawa — Precipitation Forecast (Naive Baseline) on Test Period")
plt.xlabel("Date")
plt.ylabel("Precipitation (mm)")
plt.legend()
plt.show()


## 8) Summary

What we did (Basic Assessment):
- Parsed `last_updated`, filtered to **Ottawa**, and aggregated to **daily** level
- Cleaned the time series: handled missing dates/values, clipped outliers, created a normalized target
- Ran basic EDA (trends + correlations) with required visualizations (temperature & precipitation)
- Built a basic forecasting baseline (lag-1) and evaluated it using MAE/RMSE (and MAPE for temperature)
